### 防御性+收益补偿回测

In [1]:
# -*- coding: utf-8 -*-
"""固定 SVM 因子的防御性市值分组回测：直接粘贴到 BigQuant Notebook 一个单元运行。"""



import importlib.util
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================== 回测参数 ==============================
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", "/home/aiuser/work"))
MODEL_ARTIFACT_DIR = (
    PROJECT_ROOT
    / "factor_lib"
    / "model_artifacts"
    / "svm_model_bundles"
    / "svm_all_a_fixed_20260831_223116_5bfc17b8"
)

# 模型训练的信息截止日是 2025-02-14；严格样本外回测从其后开始。
BACKTEST_START_DATE = "2025-02-18"
BACKTEST_END_DATE = "2026-08-28"
REBALANCE_INTERVAL = 20

# 五等分市值，仅使用第 1 组（最小市值组）；在该组内买入 svm_score 最高的 5%。
MARKET_CAP_GROUP_COUNT = 5
SELECTED_MARKET_CAP_GROUPS = [1]
FACTOR_QUANTILE_RANGE = (0.95, 1.00)

# 回测表现基准与防御触发基准均为中证 2000（932000）。
BACKTEST_BENCHMARK = "932000.CSI"
DEFENSIVE_BENCHMARK_INDEX = "csi_2000"
DEFENSIVE_MA_WINDOW = 60
# 中证 2000 的信号日收盘价低于 MA60 时，原 SVM 组合仅保留 5% 仓位。
DEFENSIVE_STRATEGY_WEIGHT = 0.05
# 可自行修改；触发防御后其余 95% 按此列表等权配置。
DEFENSIVE_COMPENSATION_INSTRUMENTS = [
    "601398.SH",
    "601328.SH",
    "601988.SH",
]

INITIAL_CASH = 1_000_000
TRADING_COSTS = {
    "buy_cost": 0.0003,
    "sell_cost": 0.0003,
    "min_cost": 5.0,
    "tax_ratio": 0.0005,
}
SLIPPAGE_VALUE = 0.001
VOLUME_LIMIT = 0.025
PROGRESS_EVERY = 1


def _find_project_root(root_hint):
    root_hint = Path(root_hint).expanduser()
    candidates = [root_hint, *root_hint.parents, Path.cwd(), *Path.cwd().parents]
    project_root = next(
        (item.resolve() for item in candidates if (item / "factor_lib").is_dir()),
        None,
    )
    if project_root is None:
        raise FileNotFoundError(
            "未找到项目根目录；BigQuant 环境通常应为 /home/aiuser/work。"
        )
    return project_root


def _load_module(module_name, module_path):
    if not module_path.is_file():
        raise FileNotFoundError(f"找不到脚本：{module_path}")
    module_spec = importlib.util.spec_from_file_location(module_name, module_path)
    if module_spec is None or module_spec.loader is None:
        raise ImportError(f"无法加载脚本：{module_path}")
    module = importlib.util.module_from_spec(module_spec)
    sys.modules[module_name] = module
    module_spec.loader.exec_module(module)
    return module


PROJECT_ROOT = _find_project_root(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if not MODEL_ARTIFACT_DIR.is_dir():
    raise FileNotFoundError(f"找不到模型包目录：{MODEL_ARTIFACT_DIR}")

SVM_MODULE_PATH = (
    PROJECT_ROOT
    / "factor_lib"
    / "Factor Repository"
    / "machine_learning_factors"
    / "svm_score.py"
)
svm_module = _load_module("svm_score_defensive_backtest", SVM_MODULE_PATH)
model_bundle = svm_module.load_svm_model_bundle(MODEL_ARTIFACT_DIR)

from factor_lib.function.bigquant_function.strategies.market_cap_group_backtest import (  # noqa: E402
    run_defensive_market_cap_group_backtest,
)


# 在同一 Notebook 内保留已经推理完成的单日 SVM 截面；调整回测参数时不会
# 重复读取十个依赖特征，且每次推理仍只使用相应信号日及此前的数据。
SVM_SCORE_CACHE_KEY = (str(MODEL_ARTIFACT_DIR.resolve()), "all_a")
_notebook_globals = globals()
_score_panel_cache = _notebook_globals.get("_SVM_SCORE_PANEL_CACHE")
if _score_panel_cache is None:
    _score_panel_cache = {}
    _notebook_globals["_SVM_SCORE_PANEL_CACHE"] = _score_panel_cache
if not isinstance(_score_panel_cache, dict):
    raise TypeError("_SVM_SCORE_PANEL_CACHE 必须是字典；请重启内核后重试。")
_score_cache_by_date = _score_panel_cache.setdefault(
    SVM_SCORE_CACHE_KEY,
    {},
)


def svm_factor_panel_provider(signal_dates):
    """按信号日流式推理 SVM，避免一次性加载全部特征历史导致内存不足。"""
    dates = (
        pd.DatetimeIndex(pd.to_datetime(signal_dates))
        .normalize()
        .unique()
        .sort_values()
    )
    started_at = time.perf_counter()
    missing_dates = [
        signal_date
        for signal_date in dates
        if signal_date not in _score_cache_by_date
    ]
    print(
        f"[防御性 SVM 市值分组回测] 需要 {len(dates)} 个信号日评分；"
        f"缓存命中 {len(dates) - len(missing_dates)} 个，"
        f"待流式推理 {len(missing_dates)} 个。",
        flush=True,
    )
    for position, signal_date in enumerate(missing_dates, start=1):
        print(
            f"[防御性 SVM 市值分组回测] 流式推理 "
            f"{position}/{len(missing_dates)} | 当前 {signal_date:%Y-%m-%d} "
            f"| 已耗时 {time.perf_counter() - started_at:.1f}s",
            flush=True,
        )
        score = svm_module.infer_svm_score(
            target_dates=[signal_date],
            model_bundle=model_bundle,
            universe={"type": "all_a"},
            as_of_date=signal_date,
            batch_signal_dates=1,
            show_progress=True,
            progress_every=1,
        )
        _score_cache_by_date[signal_date] = score.loc[
            :, ["date", "instrument", "svm_score"]
        ].copy()

    score_panel = pd.concat(
        [_score_cache_by_date[signal_date] for signal_date in dates],
        axis=0,
        ignore_index=True,
    )
    score_panel["date"] = pd.to_datetime(score_panel["date"]).dt.normalize()
    return score_panel.drop_duplicates(["date", "instrument"])


defensive_svm_backtest_result = run_defensive_market_cap_group_backtest(
    start_date=BACKTEST_START_DATE,
    end_date=BACKTEST_END_DATE,
    rebalance_interval=REBALANCE_INTERVAL,
    universe={"type": "all_a"},
    factor_name="svm_score",
    market_cap_group_count=MARKET_CAP_GROUP_COUNT,
    selected_market_cap_groups=SELECTED_MARKET_CAP_GROUPS,
    factor_quantile_range=FACTOR_QUANTILE_RANGE,
    factor_params=None,
    factor_panel_provider=svm_factor_panel_provider,
    defensive_benchmark_index=DEFENSIVE_BENCHMARK_INDEX,
    defensive_ma_window=DEFENSIVE_MA_WINDOW,
    defensive_strategy_weight=DEFENSIVE_STRATEGY_WEIGHT,
    defensive_compensation_instruments=DEFENSIVE_COMPENSATION_INSTRUMENTS,
    order_price_field_buy="open",
    order_price_field_sell="open",
    initial_cash=INITIAL_CASH,
    benchmark=BACKTEST_BENCHMARK,
    trading_costs=TRADING_COSTS,
    slippage_value=SLIPPAGE_VALUE,
    volume_limit=VOLUME_LIMIT,
    show_progress=True,
    progress_every=PROGRESS_EVERY,
)

display(pd.DataFrame([defensive_svm_backtest_result["data_diagnostics"]]))
display(defensive_svm_backtest_result["rebalance_audit"].tail(10))
defensive_svm_backtest_result["performance"]


ImportError: cannot import name 'run_defensive_market_cap_group_backtest' from 'factor_lib.function.bigquant_function.strategies.market_cap_group_backtest' (/home/aiuser/work/factor_lib/function/bigquant_function/strategies/market_cap_group_backtest.py)